# SQL 业务查询与指标验证

本部分使用 SQL 对前面 Python 生成的 `data/processed` 分析结果进行业务查询和指标验证。

Python 主要承担大文件处理、分块计算、复杂特征构建和可视化；SQL 主要承担结构化查询、指标复现、分群汇总和业务口径验证。在真实数据分析工作中，SQL 是日常取数和指标验证的核心技能，因此本项目补充 SQL 模块，以体现完整的数据分析工作流。

注意：本 Notebook 不读取原始 3.67GB `UserBehavior.csv`，也不重新计算全量指标，只导入 `data/processed` 下已经生成的结果表。


## 一、导入库

导入 pandas、sqlite3 和 os。


In [ ]:
# 导入数据处理、SQLite 数据库和路径处理工具
import pandas as pd
import sqlite3
import os


## 二、设置路径

设置项目路径、processed 目录和 SQLite 数据库路径。


In [ ]:
# 项目根目录
project_path = r"E:\ecommerce-user-growth-analysis"

# processed 结果表目录
processed_path = r"E:\ecommerce-user-growth-analysis\data\processed"

# SQLite 数据库保存路径
db_path = r"E:\ecommerce-user-growth-analysis\data\processed\ecommerce_analysis.db"

# 确保 processed 目录存在
os.makedirs(processed_path, exist_ok=True)

print("项目路径：", project_path)
print("processed 路径：", processed_path)
print("SQLite 数据库路径：", db_path)


## 三、连接 SQLite

使用 `sqlite3.connect` 创建数据库连接。


In [ ]:
# 创建 SQLite 数据库连接；如果数据库文件不存在，sqlite3 会自动创建
conn = sqlite3.connect(db_path)

print("SQLite 数据库连接已创建")


## 四、自动导入 processed CSV

遍历指定 CSV 文件，如果文件存在就导入 SQLite；如果不存在，只打印提示，不中断程序。


In [ ]:
# CSV 文件名和 SQLite 表名的映射关系
csv_table_map = {
    "full_core_metrics.csv": "full_core_metrics",
    "behavior_funnel_metrics.csv": "behavior_funnel_metrics",
    "user_funnel_metrics.csv": "user_funnel_metrics",
    "daily_activity.csv": "daily_activity",
    "hourly_activity.csv": "hourly_activity",
    "weekday_activity.csv": "weekday_activity",
    "user_retention.csv": "user_retention",
    "user_segmentation.csv": "user_segmentation",
    "user_segment_summary.csv": "user_segment_summary",
    "category_metrics.csv": "category_metrics",
    "category_opportunity.csv": "category_opportunity",
    "category_strategy_recommendations.csv": "category_strategy_recommendations",
}


def enrich_category_opportunity(df):
    """为 category_opportunity 补充 SQL 查询需要的缺口字段。"""
    # 如果字段存在，就直接返回，避免重复计算覆盖已有字段
    if "pv_no_buy_user_gap" not in df.columns and {"pv_user_count", "buy_user_count"}.issubset(df.columns):
        df["pv_no_buy_user_gap"] = df["pv_user_count"] - df["buy_user_count"]

    if "cart_no_buy_user_gap" not in df.columns and {"cart_user_count", "buy_user_count"}.issubset(df.columns):
        df["cart_no_buy_user_gap"] = df["cart_user_count"] - df["buy_user_count"]

    if "fav_no_buy_user_gap" not in df.columns and {"fav_user_count", "buy_user_count"}.issubset(df.columns):
        df["fav_no_buy_user_gap"] = df["fav_user_count"] - df["buy_user_count"]

    if "pv_no_buy_rate" not in df.columns and {"pv_no_buy_user_gap", "pv_user_count"}.issubset(df.columns):
        df["pv_no_buy_rate"] = df.apply(
            lambda x: x["pv_no_buy_user_gap"] / x["pv_user_count"] if x["pv_user_count"] else 0,
            axis=1,
        )

    if "cart_no_buy_rate" not in df.columns and {"cart_no_buy_user_gap", "cart_user_count"}.issubset(df.columns):
        df["cart_no_buy_rate"] = df.apply(
            lambda x: x["cart_no_buy_user_gap"] / x["cart_user_count"] if x["cart_user_count"] else 0,
            axis=1,
        )

    if "fav_no_buy_rate" not in df.columns and {"fav_no_buy_user_gap", "fav_user_count"}.issubset(df.columns):
        df["fav_no_buy_rate"] = df.apply(
            lambda x: x["fav_no_buy_user_gap"] / x["fav_user_count"] if x["fav_user_count"] else 0,
            axis=1,
        )

    return df


# 遍历 processed 目录中的结果表，存在则导入 SQLite，不存在则提示并跳过
for csv_file, table_name in csv_table_map.items():
    csv_path = os.path.join(processed_path, csv_file)

    if os.path.exists(csv_path):
        # 读取 CSV 结果表
        df = pd.read_csv(csv_path)

        # category_opportunity 的部分 SQL 查询需要 gap/rate 字段，这里基于已有字段补充
        if table_name == "category_opportunity":
            df = enrich_category_opportunity(df)

        # 导入 SQLite，replace 表示如果表已存在就覆盖
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"已导入 {csv_file} 到 {table_name} 表，行数：{len(df):,}")
    else:
        print(f"文件不存在，跳过 {csv_file}")


## 五、查看数据库中已有表

通过查询 `sqlite_master` 查看当前 SQLite 数据库中有哪些表。


In [ ]:
# 查询 SQLite 当前已有表
tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn,
)

display(tables)


## 六、SQL 查询执行函数

封装一个查询函数：如果依赖表缺失或字段不匹配，会打印提示和字段信息，便于初学者调试。


In [ ]:
# 定义一个安全执行 SQL 的函数
def run_sql(query, explanation, required_tables=None):
    """执行 SQL 查询；如果表或字段不存在，打印提示，不中断 Notebook。"""
    required_tables = required_tables or []

    # 查询当前数据库已有表
    existing_tables = set(
        pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type = 'table';",
            conn,
        )["name"]
    )

    # 如果依赖表不存在，直接提示并跳过
    missing_tables = [table for table in required_tables if table not in existing_tables]
    if missing_tables:
        print(f"跳过查询：缺少表 {missing_tables}")
        print("业务解释：", explanation)
        return None

    try:
        result = pd.read_sql_query(query, conn)
        display(result)
        print("业务解释：", explanation)
        return result
    except Exception as error:
        print("SQL 执行失败：", error)
        print("建议先检查相关表字段：")
        for table in required_tables:
            try:
                columns = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
                print(f"\n{table} 字段：")
                display(columns[["name", "type"]])
            except Exception as column_error:
                print(f"无法读取 {table} 字段：", column_error)
        print("业务解释：", explanation)
        return None


## 查询 1：查看全量核心指标

```sql
SELECT *
FROM full_core_metrics;
```


In [ ]:
# 查询 1：查看全量核心指标
query = """SELECT *
FROM full_core_metrics;"""

run_sql(
    query,
    explanation="用于查看平台整体流量规模、用户规模、商品规模、购买规模和用户购买覆盖率。",
    required_tables=['full_core_metrics'],
)


## 查询 2：查看行为漏斗结果

```sql
SELECT *
FROM behavior_funnel_metrics;
```


In [ ]:
# 查询 2：查看行为漏斗结果
query = """SELECT *
FROM behavior_funnel_metrics;"""

run_sql(
    query,
    explanation="用于查看浏览、收藏、加购、购买之间的行为转化关系。",
    required_tables=['behavior_funnel_metrics'],
)


## 查询 3：查看用户漏斗结果

```sql
SELECT *
FROM user_funnel_metrics;
```


In [ ]:
# 查询 3：查看用户漏斗结果
query = """SELECT *
FROM user_funnel_metrics;"""

run_sql(
    query,
    explanation="用于查看不同用户行为阶段的人数变化和用户层面的转化情况。",
    required_tables=['user_funnel_metrics'],
)


## 查询 4：查看用户分群汇总

```sql
SELECT *
FROM user_segment_summary;
```


In [ ]:
# 查询 4：查看用户分群汇总
query = """SELECT *
FROM user_segment_summary;"""

run_sql(
    query,
    explanation="用于查看不同用户群体的规模、占比和平均行为表现。",
    required_tables=['user_segment_summary'],
)


## 查询 5：基于 user_segmentation 查询各用户群体人数和占比

```sql
SELECT
    user_segment,
    COUNT(*) AS user_count,
    ROUND(COUNT(*) * 1.0 / (SELECT COUNT(*) FROM user_segmentation), 4) AS user_ratio
FROM user_segmentation
GROUP BY user_segment
ORDER BY user_count DESC;
```


In [ ]:
# 查询 5：基于 user_segmentation 查询各用户群体人数和占比
query = """SELECT
    user_segment,
    COUNT(*) AS user_count,
    ROUND(COUNT(*) * 1.0 / (SELECT COUNT(*) FROM user_segmentation), 4) AS user_ratio
FROM user_segmentation
GROUP BY user_segment
ORDER BY user_count DESC;"""

run_sql(
    query,
    explanation="用于判断平台主要用户结构，识别哪些用户群体是运营重点。",
    required_tables=['user_segmentation'],
)


## 查询 6：查询各用户群体平均行为表现

```sql
SELECT
    user_segment,
    ROUND(AVG(total_behavior_count), 2) AS avg_total_behavior,
    ROUND(AVG(pv_count), 2) AS avg_pv,
    ROUND(AVG(fav_count), 2) AS avg_fav,
    ROUND(AVG(cart_count), 2) AS avg_cart,
    ROUND(AVG(buy_count), 2) AS avg_buy,
    ROUND(AVG(active_days), 2) AS avg_active_days
FROM user_segmentation
GROUP BY user_segment
ORDER BY avg_buy DESC;
```


In [ ]:
# 查询 6：查询各用户群体平均行为表现
query = """SELECT
    user_segment,
    ROUND(AVG(total_behavior_count), 2) AS avg_total_behavior,
    ROUND(AVG(pv_count), 2) AS avg_pv,
    ROUND(AVG(fav_count), 2) AS avg_fav,
    ROUND(AVG(cart_count), 2) AS avg_cart,
    ROUND(AVG(buy_count), 2) AS avg_buy,
    ROUND(AVG(active_days), 2) AS avg_active_days
FROM user_segmentation
GROUP BY user_segment
ORDER BY avg_buy DESC;"""

run_sql(
    query,
    explanation="用于比较不同用户群体的活跃度、购买能力和运营价值。",
    required_tables=['user_segmentation'],
)


## 查询 7：查询加购未购买用户数量

```sql
SELECT
    COUNT(*) AS cart_no_buy_user_count
FROM user_segmentation
WHERE cart_count > 0 AND buy_count = 0;
```


In [ ]:
# 查询 7：查询加购未购买用户数量
query = """SELECT
    COUNT(*) AS cart_no_buy_user_count
FROM user_segmentation
WHERE cart_count > 0 AND buy_count = 0;"""

run_sql(
    query,
    explanation="这类用户已有较强购买意向，适合通过优惠券、降价提醒、库存提醒促进转化。",
    required_tables=['user_segmentation'],
)


## 查询 8：查询收藏未购买用户数量

```sql
SELECT
    COUNT(*) AS fav_no_buy_user_count
FROM user_segmentation
WHERE fav_count > 0 AND buy_count = 0;
```


In [ ]:
# 查询 8：查询收藏未购买用户数量
query = """SELECT
    COUNT(*) AS fav_no_buy_user_count
FROM user_segmentation
WHERE fav_count > 0 AND buy_count = 0;"""

run_sql(
    query,
    explanation="这类用户对商品有兴趣但决策较慢，适合推送评价、买家秀、相似商品和降价提醒。",
    required_tables=['user_segmentation'],
)


## 查询 9：查询高浏览低购买用户 Top 20

```sql
SELECT
    user_id,
    pv_count,
    fav_count,
    cart_count,
    buy_count,
    total_behavior_count,
    active_days
FROM user_segmentation
WHERE buy_count = 0
ORDER BY pv_count DESC
LIMIT 20;
```


In [ ]:
# 查询 9：查询高浏览低购买用户 Top 20
query = """SELECT
    user_id,
    pv_count,
    fav_count,
    cart_count,
    buy_count,
    total_behavior_count,
    active_days
FROM user_segmentation
WHERE buy_count = 0
ORDER BY pv_count DESC
LIMIT 20;"""

run_sql(
    query,
    explanation="用于识别浏览很多但没有购买的用户，后续可优化推荐精准度和转化刺激策略。",
    required_tables=['user_segmentation'],
)


## 查询 10：查看每日活跃趋势

```sql
SELECT *
FROM daily_activity
ORDER BY date
LIMIT 20;
```


In [ ]:
# 查询 10：查看每日活跃趋势
query = """SELECT *
FROM daily_activity
ORDER BY date
LIMIT 20;"""

run_sql(
    query,
    explanation="用于观察平台每日行为量、活跃用户和购买行为的变化。",
    required_tables=['daily_activity'],
)


## 查询 11：查看每小时活跃趋势

```sql
SELECT *
FROM hourly_activity
ORDER BY hour;
```


In [ ]:
# 查询 11：查看每小时活跃趋势
query = """SELECT *
FROM hourly_activity
ORDER BY hour;"""

run_sql(
    query,
    explanation="用于识别用户一天中的活跃高峰，为推荐、优惠券和活动推送时间提供依据。",
    required_tables=['hourly_activity'],
)


## 查询 12：查看类目机会类型分布

```sql
SELECT
    opportunity_type,
    COUNT(DISTINCT category_id) AS category_count
FROM category_opportunity
GROUP BY opportunity_type
ORDER BY category_count DESC;
```


In [ ]:
# 查询 12：查看类目机会类型分布
query = """SELECT
    opportunity_type,
    COUNT(DISTINCT category_id) AS category_count
FROM category_opportunity
GROUP BY opportunity_type
ORDER BY category_count DESC;"""

run_sql(
    query,
    explanation="用于查看不同类型增长机会类目的数量分布。",
    required_tables=['category_opportunity'],
)


## 查询 13：查看高加购低购买类目 Top 20

```sql
SELECT
    category_id,
    opportunity_type,
    cart_count,
    buy_count,
    cart_user_count,
    buy_user_count,
    cart_no_buy_user_gap,
    cart_no_buy_rate
FROM category_opportunity
WHERE opportunity_type = '高加购低购买类目'
ORDER BY cart_no_buy_user_gap DESC
LIMIT 20;
```


In [ ]:
# 查询 13：查看高加购低购买类目 Top 20
query = """SELECT
    category_id,
    opportunity_type,
    cart_count,
    buy_count,
    cart_user_count,
    buy_user_count,
    cart_no_buy_user_gap,
    cart_no_buy_rate
FROM category_opportunity
WHERE opportunity_type = '高加购低购买类目'
ORDER BY cart_no_buy_user_gap DESC
LIMIT 20;"""

run_sql(
    query,
    explanation="用于识别已经有大量用户加购但未购买的类目，适合做优惠刺激和临门一脚转化。",
    required_tables=['category_opportunity'],
)


## 查询 14：查看高浏览低购买类目 Top 20

```sql
SELECT
    category_id,
    opportunity_type,
    pv_count,
    buy_count,
    pv_user_count,
    buy_user_count,
    pv_no_buy_user_gap,
    pv_no_buy_rate
FROM category_opportunity
WHERE opportunity_type = '高浏览低购买类目'
ORDER BY pv_no_buy_user_gap DESC
LIMIT 20;
```


In [ ]:
# 查询 14：查看高浏览低购买类目 Top 20
query = """SELECT
    category_id,
    opportunity_type,
    pv_count,
    buy_count,
    pv_user_count,
    buy_user_count,
    pv_no_buy_user_gap,
    pv_no_buy_rate
FROM category_opportunity
WHERE opportunity_type = '高浏览低购买类目'
ORDER BY pv_no_buy_user_gap DESC
LIMIT 20;"""

run_sql(
    query,
    explanation="用于识别流量高但转化不足的类目，适合优化详情页、价格展示、评价内容和推荐精准度。",
    required_tables=['category_opportunity'],
)


## 查询 15：查看类目策略建议

```sql
SELECT
    category_id,
    opportunity_type,
    business_problem,
    strategy_recommendation
FROM category_strategy_recommendations
LIMIT 20;
```


In [ ]:
# 查询 15：查看类目策略建议
query = """SELECT
    category_id,
    opportunity_type,
    business_problem,
    strategy_recommendation
FROM category_strategy_recommendations
LIMIT 20;"""

run_sql(
    query,
    explanation="用于把类目机会转化为可落地的产品运营策略。",
    required_tables=['category_strategy_recommendations'],
)


## 七、关闭数据库连接

所有查询执行完成后，关闭 SQLite 连接，释放数据库资源。


In [ ]:
# 关闭 SQLite 数据库连接
conn.close()

print("SQLite 数据库连接已关闭")
